# Notebook 02 – Baseline Models

Before training LSTM models, we establish a **simple baseline** using **Linear Regression**.
This provides a lower bound on performance: if our optimised LSTM cannot beat linear regression, there is a fundamental problem with the approach.

We also measure:
- Training time
- Prediction MAPE
- Model size

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

from src.data_preprocessing import load_and_preprocess
from src.evaluate import mean_absolute_percentage_error

%matplotlib inline

## 1. Load Preprocessed Data

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test, scaler = load_and_preprocess(
    csv_path='../data/raw/train.csv',
    window_size=30,
)

# Flatten window dimension for sklearn
X_tr_flat = X_train.reshape(len(X_train), -1)
X_te_flat = X_test.reshape(len(X_test),  -1)

print('Train:', X_tr_flat.shape, '  Test:', X_te_flat.shape)

## 2. Linear Regression

In [ ]:
t0 = time.perf_counter()
lr_model = LinearRegression()
lr_model.fit(X_tr_flat, y_train)
train_time = time.perf_counter() - t0

y_pred_scaled = lr_model.predict(X_te_flat)

# Inverse-transform
y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

mse  = mean_squared_error(y_true, y_pred)
mape = mean_absolute_percentage_error(y_true, y_pred)

print(f'Train time : {train_time*1000:.1f} ms')
print(f'Test MSE   : {mse:.4f}')
print(f'Test MAPE  : {mape:.2f} %')

## 3. Visualise Predictions vs Actuals

In [ ]:
n_show = 200
plt.figure(figsize=(14, 4))
plt.plot(y_true[:n_show],  label='Actual',  color='steelblue')
plt.plot(y_pred[:n_show],  label='LR Pred', color='orange', linestyle='--')
plt.title('Linear Regression – First 200 Test Predictions')
plt.xlabel('Sample index')
plt.ylabel('Sales')
plt.legend()
plt.tight_layout()
plt.savefig('../results/baseline_lr_predictions.png', dpi=120)
plt.show()

## 4. Baseline Summary

In [ ]:
summary = pd.DataFrame([{
    'model':        'Linear Regression',
    'n_parameters': X_tr_flat.shape[1] + 1,  # coefficients + intercept
    'mse':          round(mse, 4),
    'mape_pct':     round(mape, 4),
    'train_time_ms': round(train_time * 1000, 2),
}])

os.makedirs('../results', exist_ok=True)
summary.to_csv('../results/baseline_summary.csv', index=False)
summary